Excellent. This notebook introduces the core concepts behind RAG (Retrieval-Augmented Generation). Once you understand this notebook, you'll understand how ChatGPT can answer questions using your own documents instead of only its training data.

I'll simplify it, remove unnecessary code, explain every concept, and organize it in the order you should learn.

What are we building?

We are building a very simple RAG pipeline.

Instead of asking only the LLM:

Question
    │
    ▼
LLM
    │
    ▼
Answer

We first search our documents.

Question
    │
    ▼
Retriever
    │
    ▼
Relevant Documents
    │
    ▼
LLM
    │
    ▼
Answer

The LLM answers using the retrieved documents.

Overall Architecture
Documents
     │
     ▼
Embeddings
     │
     ▼
Vector Store
     │
     ▼
Retriever
     │
     ▼
Relevant Documents
     │
     ▼
Prompt Template
     │
     ▼
LLM
     │
     ▼
Answer
Before Starting
Install Libraries

In [1]:
!pip install langchain
!pip install langchain-groq
!pip install langchain-chroma
!pip install langchain-huggingface
!pip install python-dotenv


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
#Step 1 Load Environment Variables
import os
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

# We use the Groq API key for the LLM.

In [3]:
#Step 2 Create the LLM
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama3-8b-8192",
    groq_api_key=groq_api_key
)

c:\DS2026\Python\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
#Step 3 Understanding Documents

from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are loyal.",
        metadata={"source":"pets"}
    ),

    Document(
        page_content="Cats like sleeping.",
        metadata={"source":"pets"}
    ),

    Document(
        page_content="Goldfish live in water.",
        metadata={"source":"fish"}
    )
]

In [5]:
#Create Embedding Model

from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 918.48it/s]


In [6]:
#Step 5 Vector Store

from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents,
    embedding=embeddings
)

In [7]:
#Step 6 Search

#Now we search.

vectorstore.similarity_search("cat")

[Document(id='30d470ba-1080-49e7-be93-fef46db2c36a', metadata={'source': 'pets'}, page_content='Cats like sleeping.'),
 Document(id='a929949c-1171-430c-88d7-2ad5414ad258', metadata={'source': 'pets'}, page_content='Dogs are loyal.'),
 Document(id='48e32953-3e12-4f21-8a8c-b24664f7853c', metadata={'source': 'fish'}, page_content='Goldfish live in water.')]

In [8]:
#similarity_search()

#Returns only documents.

results = vectorstore.similarity_search("dog")

In [9]:
#Async Search
await vectorstore.asimilarity_search("cat")

[Document(id='30d470ba-1080-49e7-be93-fef46db2c36a', metadata={'source': 'pets'}, page_content='Cats like sleeping.'),
 Document(id='a929949c-1171-430c-88d7-2ad5414ad258', metadata={'source': 'pets'}, page_content='Dogs are loyal.'),
 Document(id='48e32953-3e12-4f21-8a8c-b24664f7853c', metadata={'source': 'fish'}, page_content='Goldfish live in water.')]

Step 7 Retriever

This is the second most important concept.

Many databases exist.

Chroma
Pinecone
FAISS
Weaviate
Milvus

Each has different APIs.

LangChain hides these differences using a Retriever.

Instead of calling the vector database directly,

your chain talks to a retriever.

Question
      │
      ▼
Retriever
      │
      ▼
Vector Store
Why Do We Need a Retriever?

Because LCEL chains work with Runnables, and a VectorStore is not a Runnable.

A Retriever wraps the VectorStore and makes it compatible with LCEL.

Manual Retriever (Can Skip)

In [10]:
from langchain_core.runnables import RunnableLambda

retriever = RunnableLambda(
    vectorstore.similarity_search
).bind(k=1)

In [11]:
#This simply wraps the similarity_search() method as a Runnable.

#For beginners, you can skip this approach.

#Recommended Way
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":1}
)

#This creates a VectorStoreRetriever.

In [12]:
#Step 8 Test Retriever
retriever.invoke("dog")

[Document(id='a929949c-1171-430c-88d7-2ad5414ad258', metadata={'source': 'pets'}, page_content='Dogs are loyal.')]

In [13]:
retriever.batch([
    "dog",
    "cat"
])

[[Document(id='a929949c-1171-430c-88d7-2ad5414ad258', metadata={'source': 'pets'}, page_content='Dogs are loyal.')],
 [Document(id='30d470ba-1080-49e7-be93-fef46db2c36a', metadata={'source': 'pets'}, page_content='Cats like sleeping.')]]

In [14]:
#Step 9 Prompt Template

#Now we give both the question and the retrieved context to the LLM.

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
[
(
"human",
"""
Answer only using the context.

Question:
{question}

Context:
{context}
"""
)
]
)

#This creates a structured prompt.

In [15]:
#Step 10 RunnablePassthrough
from langchain_core.runnables import RunnablePassthrough

#This simply passes the user's question unchanged through the chain.

In [16]:
#Step 11 Build the RAG Chain (LCEL)
rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)

In [17]:
#Step 12 Ask a Question
response = rag_chain.invoke(
    "Tell me about dogs"
)

print(response.content)

BadRequestError: Error code: 400 - {'error': {'message': 'The model `llama3-8b-8192` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}

Execution flow:

Question
      │
      ▼
Retriever
      │
      ▼
Relevant Documents
      │
      ▼
Prompt Template
      │
      ▼
LLM
      │
      ▼
Answer
Code You Can Remove

For learning purposes, you can safely remove:

await vectorstore.asimilarity_search(...) (advanced asynchronous example).
The manual Retriever built with RunnableLambda; use vectorstore.as_retriever() instead.
retriever.batch(["cat", "dog"]) examples once you've verified the Retriever works.
The repeated llm object display (llm on its own line), which only shows the object representation.
Core Concepts You Must Master
Concept	Purpose
Document	Stores text and metadata together.
page_content	The actual text to search.
metadata	Information about the source (file, page, URL, etc.).
Embeddings	Convert text into numerical vectors for semantic comparison.
HuggingFaceEmbeddings	Embedding model that creates vectors from text.
Vector Store	Stores vectors and enables similarity search.
Chroma	A lightweight local vector database.
similarity_search()	Finds the most semantically similar documents.
Retriever	Standard LangChain interface for retrieving documents.
as_retriever()	Converts a VectorStore into a Runnable Retriever.
Prompt Template	Combines the user's question with retrieved context.
RunnablePassthrough	Passes the original question unchanged through the chain.
LCEL (`	`)
RAG Chain	Retrieves relevant context and then asks the LLM to answer using that context.
Learning Order for RAG

To build a strong foundation, learn these concepts in this sequence:

Document
Embeddings and why semantic vectors are needed
Vector Stores (Chroma)
Similarity Search
Retriever (as_retriever)
Prompt Templates
LCEL (|)
RunnablePassthrough
Build a simple RAG chain
Add document loaders (PDF, Word, web pages)
Add text splitting (RecursiveCharacterTextSplitter)
Build a complete RAG application with Streamlit or FastAPI
Explore advanced retrieval techniques such as MMR, reranking, and hybrid search

Once you understand this notebook, you'll have the foundation for building document Q&A systems, chatbots over PDFs, enterprise knowledge assistants, and AI agents that retrieve information before reasoning over it.